# W07B Colab A ·관측·유사도·가중평균

추천 시스템 ·동덕여자대학교 ·유원상 교수 ·2026년 10월 15일

CPU에서 위부터 실행하세요. 두 노트북은 각각 독립 실행됩니다. GPU·유료 API·필수 파일 업로드가 없습니다. **파일 → Drive에 사본 저장**으로 자신의 실습본을 만드세요. 설치 후 런타임 재시작 안내가 나오면 재시작하고 첫 셀부터 실행합니다.

문제 코드는 Markdown에 제시하고 자신의 답은 별도 셀에 씁니다. 빈칸이나 일부러 잘못된 코드를 자동으로 실행하지 않습니다. 따라서 전체 실행 성공은 시연 기능의 확인이며, 자신의 연습 문제 정답을 모두 풀었다는 뜻은 아닙니다. 설명용 데이터는 연습 문제 데이터와 별개입니다.


## 가져오는 코드의 정의를 바로 확인하기

아래 실행 셀은 설치한 고정 버전의 GitHub 파일과 **정의 시작 행**으로 연결합니다. 클래스와 메소드를 각각 확인할 수 있습니다. `help(함수)`로 입력·반환·대체 규칙을 읽어도 됩니다. 이 API를 외우는 것은 시험 문제의 요구가 아닙니다.

| 함수/클래스 | 입력 → 출력과 의미 |
|---|---|
| toy_ratings | 인자 없음 → 새 합성 관측18행. 없는 행은 미관측 |
| binary_jaccard | 두 행동 집합 → 교집합/합집합, 둘 다 비면 근거 없음 |
| UserCF.fit | 학습 관측 → 평균·마스크·유사도를 가진 모델 |
| EvidenceCF | 축/k/ 최소 공통/ 최소 이웃/ 보정/ 범위 제한 → 설정 객체 |
| EvidenceCF.fit | train 관측 → 같은 모델 반환; SGD가 아님 |
| EvidenceCF.predict_details | 사용자·영화 ID쌍 → 예측/원예측/근거/이웃수 |
| EvidenceCF.explain | 사용자ID,영화ID → 이웃/비중/기여도 표; 대체면 빈 표 |
| rating_report / evaluate_cf | 학습된 모델, 미사용 관측 / 추가 카탈로그 → 오차·순위 지표 |
| cf_review_data | 없음 → train15/validation3/ 카탈로그 5편 |
| cf_review_experiment | 설정 → 한 예측의 근거와 동일 검증 지표 dict |
| cf_review_view | 설정, 선택 설명 함수 → 앱 출력 6개; launch하지 않음 |
| build_cf_review_app | 선택 설명 함수 → Gradio 앱; launch는 마지막 셀 |
| definition_link | 함수·클래스·메소드, 고정ref → GitHub 정의줄 URL |

fit은 데이터에서 통계를 만드는 단계입니다. 예측 대상의 실제 정답은 전달하지 않습니다. 기본 코사인은 전체 좌표의 0 채움 표현을 사용하고, Pearson의 공통 좌표 비교와 구별합니다. `axis='item',centered=True`는 이 수업에서 지원하지 않아 명시적으로 거부합니다.


In [ ]:
import os,sys,subprocess,math
os.environ['GRADIO_ANALYTICS_ENABLED']='False'
IN_COLAB='google.colab' in sys.modules
COURSE_VERSION='2026-fall-w07b'
if IN_COLAB:
    subprocess.run([sys.executable,'-m','pip','install','-q',
        f'luna-recommender-course[apps] @ git+https://github.com/lunalab-ai/recommender.git@{COURSE_VERSION}',
        'gradio==6.26.0'],check=True)
import pandas as pd
from IPython.display import display,Markdown
from luna_recsys.collaborative import toy_ratings,UserCF,binary_jaccard
from luna_recsys.cf_synthesis import EvidenceCF,evaluate_cf,rating_report
from luna_recsys.cf_review import cf_review_data,cf_review_experiment,cf_review_view,build_cf_review_app
from luna_recsys.model_comparison import definition_link
print('Python',sys.version.split()[0], '수업 버전',COURSE_VERSION)


In [ ]:
apis=[toy_ratings,binary_jaccard,UserCF,UserCF.fit,EvidenceCF,EvidenceCF.fit,
 EvidenceCF.predict_details,EvidenceCF.explain,rating_report,evaluate_cf,
 cf_review_data,cf_review_experiment,cf_review_view,build_cf_review_app,definition_link]
display(Markdown('\n'.join(f'- [{f.__qualname__}]({definition_link(f,COURSE_VERSION)})' for f in apis)))


![원본·마스크·계산용0의 구별](../../course/notion/assets/w07b/01-mask.png)

그림을 먼저 읽고 아래 코드의 행·열·분모와 연결하세요. 수업용 독자 제작 합성 예제입니다.


## 1. 관측 → 마스크 → 계산용0

원본은 None/NaN을 보존합니다. 평균은 관측 값만 계산하고,0 채움 표현은 코사인 좌표를 만드는 용도로 구별합니다. 사용자 1의 관측은 5,3,1이므로 평균 3입니다.


In [ ]:
ratings=toy_ratings()
matrix=ratings.pivot(index='user_id',columns='movie_id',values='rating')
display(matrix)
display(matrix.notna().astype(int))
display(matrix.fillna(0))
print('u1 observed mean:',matrix.loc[1].mean())


## 2. 공식에서 기초 코드로

설명 벡터 x =(3,4),y =(4,3)의 내적은 24, 길이 곱은 25입니다. 아래 각 변수를 공식의 분자·분모와 대조하세요. **확인 A1:** 두 벡터를 같은 양의 배율로 늘리면 무엇이 상쇄될까요?


In [ ]:
x=[3,4];y=[4,3]
dot=sum(x[j]*y[j] for j in range(2))
nx=sum(v*v for v in x)**0.5
ny=sum(v*v for v in y)**0.5
print('dot, lengths, cosine:',dot,nx,ny,dot/(nx*ny))
print('binary Jaccard:',binary_jaccard({'A','B'},{'B','C'}))


## 3. 한 예측을 기여도 표로 읽기

이번 시연은 원표 전체 18건을 학습에 사용해 u1이 아직 평가하지 않은 D(영화 4)를 예측합니다. 아래 검증용 실험의 train15건과 구별합니다. `weight` 합은1, `contribution` 합은 원예측입니다. **확인 A2:** 대상 영화 미평가 이웃이 분모에 남아 있어도 될까요?


In [ ]:
model=EvidenceCF(k=2).fit(ratings)
details=model.predict_details(pd.DataFrame({'user_id':[1],'movie_id':[4]}))
terms=model.explain(1,4)
display(details);display(terms)
print('weight sum:',terms.weight.sum(),'contribution sum:',terms.contribution.sum())


### LAB-A ·비중을 말로 설명하기

`explain_weight(sim,total)`에 정규화 비중을 백분율로 표현하는 문장을 작성하세요. 시연 입력은 0.6과 0.8이며 total>0입니다. 힌트: 비중 =sim/total, `f"{value:.1%}"`는 백분율 표시입니다. 자가 점검: 분모가 커지고 sim이 같으면 비중이 작아져야합니다.


In [ ]:
def explain_weight(sim,total):
    return '내가 계산한 비중과 그 의미를 여기에 적어 보세요.'
print(explain_weight(0.6,0.8))


## 4. 기 초 Python 연습 문제

문제 번호는 학생 문제 편과 동일합니다. 각 문제의 입력은 독립적입니다. 코드가 실행되는지뿐 아니라, 주어진 관측·유사도·분모·선택 조건을 지켰는지 설명하세요.


### Q40 ·관측 평균 빈칸

None은 미관측이다. None이 아닌 값만 observed에 모았다. 평균을 구하도록 빈칸을 채우고 분모의 뜻을 쓰라.

```python
row=[4,None,2,None]
observed=[]
for r in row:
    if r is not None:
        observed.append(r)
mean=sum(observed)/___
print(mean)
```

기록: 바꿀 줄 또는 빈 칸 / 변수의 중간 값 / 최종 출력 / 연결되는 이론.


In [ ]:
my_answer_Q40=''  # 자신의 풀이를 기록하고 필요하면 별도 코드 셀을 추가하세요.


### Q41 ·미관측을 0점으로 만든 오류

평점은 1 –5다. observed가 실제 관측만 포함하도록 조건 한 줄을 수정하고 평균을 쓰라. 원래 코드는 실행되지만 뜻이 잘못되었다.

```python
row=[5,None,1]
filled=[0 if r is None else r for r in row]
observed=[r for r in filled if r>=0]
print(sum(observed)/len(observed))
```

기록: 바꿀 줄 또는 빈 칸 / 변수의 중간 값 / 최종 출력 / 연결되는 이론.


In [ ]:
my_answer_Q41=''  # 자신의 풀이를 기록하고 필요하면 별도 코드 셀을 추가하세요.


### Q42 ·공통 관측 수 추적

관측은 1, 미관측은 0으로 표시했다. 코드를 실행하지 않고 common의 최종 값과 증가한 j를 쓰라.

```python
a=[1,0,1,1]
b=[0,1,1,1]
common=0
for j in range(4):
    if a[j]==1 and b[j]==1:
        common+=1
print(common)
```

기록: 바꿀 줄 또는 빈 칸 / 변수의 중간 값 / 최종 출력 / 연결되는 이론.


In [ ]:
my_answer_Q42=''  # 자신의 풀이를 기록하고 필요하면 별도 코드 셀을 추가하세요.


### Q43 ·코사인 내적 빈칸

`**2`는 제곱, `**0.5`는 제곱근이다. 코사인의 분자를 구하도록 빈칸을 채우고 출력을 쓰라.

```python
x=[1,2]
y=[2,1]
dot=0
for j in range(2):
    dot+=___
nx=sum(v**2 for v in x)**0.5
ny=sum(v**2 for v in y)**0.5
sim=dot/(nx*ny)
print(round(sim,4))
```

기록: 바꿀 줄 또는 빈 칸 / 변수의 중간 값 / 최종 출력 / 연결되는 이론.


In [ ]:
my_answer_Q43=''  # 자신의 풀이를 기록하고 필요하면 별도 코드 셀을 추가하세요.


### Q44 ·Pearson 편차 빈칸

각 평균을 빼는 빈칸을 채우고 출력되는 두 리스트를 쓰라. 이 코드는 유사도 계산의 준비 단계다.

```python
x=[2,3,4]
y=[1,3,5]
mx=sum(x)/len(x)
my=sum(y)/len(y)
dx=[v-___ for v in x]
dy=[v-___ for v in y]
print(dx,dy)
```

기록: 바꿀 줄 또는 빈 칸 / 변수의 중간 값 / 최종 출력 / 연결되는 이론.


In [ ]:
my_answer_Q44=''  # 자신의 풀이를 기록하고 필요하면 별도 코드 셀을 추가하세요.


### Q45 ·정의 불가 분모의 처리

nx 또는 ny가 0일 때는 None(근거 없음), 그 외에는 코사인을 반환하려고 한다. 오류 원인을 설명하고 조건 문을 수정하라.

```python
dot=0
nx=0
ny=5
if nx==0 and ny==0:
    sim=None
else:
    sim=dot/(nx*ny)
print(sim)
```

기록: 바꿀 줄 또는 빈 칸 / 변수의 중간 값 / 최종 출력 / 연결되는 이론.


In [ ]:
my_answer_Q45=''  # 자신의 풀이를 기록하고 필요하면 별도 코드 셀을 추가하세요.


### Q46 ·행동 집합 코드 추적

A와 B에는 중복 없는 행동 영화가 들어 있다. 합집합을 직접 만드는 코드다. intersection,union, 출력 값을 쓰라.

```python
a=['A','C','D']
b=['B','C','D']
intersection=[]
union=a.copy()
for item in b:
    if item in a:
        intersection.append(item)
    if item not in union:
        union.append(item)
print(intersection,union,len(intersection)/len(union))
```

기록: 바꿀 줄 또는 빈 칸 / 변수의 중간 값 / 최종 출력 / 연결되는 이론.


In [ ]:
my_answer_Q46=''  # 자신의 풀이를 기록하고 필요하면 별도 코드 셀을 추가하세요.


### Q47 ·가중평균 분모 빈 칸

같은 순서의 유사도와 평점이다. 분모를 누적하는 빈칸을 채우고 예측을 구하라.

```python
sims=[0.2,0.8]
ratings=[2,5]
num=0
den=0
for j in range(2):
    num+=sims[j]*ratings[j]
    den+=___
pred=num/den
print(pred)
```

기록: 바꿀 줄 또는 빈 칸 / 변수의 중간 값 / 최종 출력 / 연결되는 이론.


In [ ]:
my_answer_Q47=''  # 자신의 풀이를 기록하고 필요하면 별도 코드 셀을 추가하세요.


### Q48 ·미평가 이웃의 유사도가 분모에 남는 오류

None은 목표 영화 미평가다. 분자와 분모가 같은 관측 이웃을 쓰도록 들여쓰기를 수정하고 출력을 쓰라.

```python
sims=[0.9,0.5,0.5]
ratings=[None,2,4]
num=0
den=0
for j in range(3):
    if ratings[j] is not None:
        num+=sims[j]*ratings[j]
    den+=sims[j]
print(round(num/den,4))
```

기록: 바꿀 줄 또는 빈 칸 / 변수의 중간 값 / 최종 출력 / 연결되는 이론.


In [ ]:
my_answer_Q48=''  # 자신의 풀이를 기록하고 필요하면 별도 코드 셀을 추가하세요.


### Q49 ·유효 이웃 조건 빈칸

평점이 존재하고 유사도가 양수인 경우만 사용한다. 빈칸을 채우고 예측을 구하라. and는 두 조건 모두 참이어야한다.

```python
rows=[(0.7,4),(0.0,5),(-0.2,1),(0.3,None)]
num=0
den=0
for sim,rating in rows:
    if rating is not None and ___:
        num+=sim*rating
        den+=sim
pred=num/den
print(pred)
```

기록: 바꿀 줄 또는 빈 칸 / 변수의 중간 값 / 최종 출력 / 연결되는 이론.


In [ ]:
my_answer_Q49=''  # 자신의 풀이를 기록하고 필요하면 별도 코드 셀을 추가하세요.


### Q50 ·분모 0의 대체 분기

유효 이웃이 없으면 영화 평균을 쓴다. 원래 코드의 계산 순서를 수정하여 오류 없이 의도대로 출력하라.

```python
num=0
den=0
movie_mean=3.5
pred=num/den
if den==0:
    pred=movie_mean
print(pred)
```

기록: 바꿀 줄 또는 빈 칸 / 변수의 중간 값 / 최종 출력 / 연결되는 이론.


In [ ]:
my_answer_Q50=''  # 자신의 풀이를 기록하고 필요하면 별도 코드 셀을 추가하세요.


## 실행 결과 검증

시연의 실제 관측·예측·기여도·자료 분리를 검사합니다. 학생 연습 문제의 정답을 공개하는 검사가 아닙니다. 실패하면 오류를 확인하고 원인을 해결하세요.


In [ ]:
check_data=cf_review_data()
train_pairs=set(check_data['train'][['user_id','movie_id']].itertuples(index=False,name=None))
valid_pairs=set(check_data['validation'][['user_id','movie_id']].itertuples(index=False,name=None))
assert len(check_data['train'])==15 and len(check_data['validation'])==3
assert not train_pairs & valid_pairs
for axis,center in [('user',False),('user',True),('item',False)]:
    checked=cf_review_experiment(axis=axis,centered=center)
    terms=checked['contributions']
    base=checked['model'].user_means_.loc[1] if center else 0
    assert not terms.empty and math.isclose(terms.weight.sum(),1)
    assert math.isclose(terms.contribution.sum()+base,checked['details'].raw_prediction.iloc[0])
fallback=cf_review_experiment(min_neighbors=4)
assert fallback['contributions'].empty and fallback['details'].basis.iloc[0]=='movie-mean'
assert fallback['metrics']['rating_count']==3
print('W07B-A RUNTIME CHECK PASSED')


## 점검 질문과 가져갈 생각

1. 미관측과 0점은 왜 다른가요?
2. 이웃을 고르기 전에 목표 영화 평가 여부를 왜 확인하나요?
3. 분자와 분모에 들어가는 이웃은 어떻게 맞추나요?
4. 평균보정은 누구의 평균을 빼고 누구의 평균을 더 하나요?
5. 설정 탐색 화면을 최종 test 성과라고 부르면 왜 문제인가요?

주교재 3장과 W03 –W04 기존 수업을 복습합니다. [코사인정의](https://scikit-learn.org/stable/modules/metrics.html#cosine-similarity), [데이터누수](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage).


[다음: 독립 실행 가능한 Colab B와 마지막 앱](https://colab.research.google.com/github/lunalab-ai/recommender/blob/2026-fall-w07b/notebooks/student/w07b-cf-evidence-app.ipynb).
